# 03 — EDA final y limpieza del dataset CIC-IDS2017

**Proyecto:** VIGÍA (Sentry) — TFC
**Objetivo:** EDA definitivo sobre los 8 CSV combinados, limpieza global, análisis de fuga y generación del Parquet limpio.

**Estado del Parquet raw:** ya viene sin columnas identificadoras (`Flow_ID`, `Source_IP`, `Destination_IP`, `Timestamp`). Los nombres `Fwd_Header_Length34` y `Fwd_Header_Length55` son artefactos del renombrado de las dos columnas `Fwd Header Length` originales.

In [ ]:
# ============================================
# Setup: repo, drive, Spark
# ============================================
import os
from google.colab import drive

GITHUB_USER = "AguCS231"
REPO_NAME   = "sentry"
REPO_LOCAL  = f"/content/{REPO_NAME}"
DRIVE_PATH  = "/content/drive/MyDrive/cic-ids2017"
PARQUET_RAW   = "/content/drive/MyDrive/cic-ids2017-parquet"
PARQUET_CLEAN = "/content/drive/MyDrive/cic-ids2017-parquet-clean"

# Clonar repo si no existe
if not os.path.exists(REPO_LOCAL):
    print("Clonando repo...")
    %cd /content
    !git clone https://github.com/{GITHUB_USER}/{REPO_NAME}.git
    %cd {REPO_LOCAL}
    !git config --global user.email "srcasares2@gmail.com"
    !git config --global user.name "{GITHUB_USER}"
else:
    %cd {REPO_LOCAL}
    print(f"✅ Repo: {REPO_LOCAL}")

# Montar Drive
drive.mount('/content/drive')

# Spark
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("VIGIA_EDA_Final") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

print(f"\n✅ Spark {spark.version}")
print(f"📁 Parquet raw:   {PARQUET_RAW}  (existe: {os.path.exists(PARQUET_RAW)})")
print(f"📁 Parquet clean: {PARQUET_CLEAN} (existe: {os.path.exists(PARQUET_CLEAN)})")

## 1. Carga del dataset

Leemos del Parquet raw (rápido, ya con nombres limpios). Si no existe, hacemos fallback a los 8 CSV originales.

In [ ]:
# ============================================
# Cargar dataset (Parquet si existe, si no CSVs)
# ============================================
import os
from functools import reduce
from pyspark.sql import DataFrame

if os.path.exists(PARQUET_RAW):
    print("📥 Leyendo Parquet raw...")
    df_full = spark.read.parquet(PARQUET_RAW)
else:
    print("⚠️ Parquet raw no encontrado. Re-leyendo los 8 CSV...")
    csvs = sorted([f for f in os.listdir(DRIVE_PATH) if f.endswith(".csv")])
    dfs = []
    for archivo in csvs:
        ruta = os.path.join(DRIVE_PATH, archivo)
        print(f"  Cargando {archivo}...")
        d = spark.read.csv(ruta, header=True, inferSchema=True)
        for col_name in d.columns:
            d = d.withColumnRenamed(col_name, col_name.strip().replace(" ", "_"))
        dfs.append(d)
    df_full = reduce(DataFrame.unionByName, dfs)

    from pyspark.sql.functions import regexp_replace, trim, col
    df_full = df_full.withColumn(
        "Label", regexp_replace(col("Label"), "[^\\x00-\\x7F]", "-")
    )
    df_full = df_full.withColumn("Label", trim(col("Label")))

# Normalizar Flow_ID si viniera con espacio
if "Flow_ID" not in df_full.columns and "Flow ID" in df_full.columns:
    df_full = df_full.withColumnRenamed("Flow ID", "Flow_ID")

assert "Label" in df_full.columns, "No encuentro la columna Label"

print(f"\n✅ Dataset cargado")
print(f"   Filas:    {df_full.count():,}")
print(f"   Columnas: {len(df_full.columns)}")

# Identificadores: informar cuáles están presentes
for c in ["Flow_ID", "Source_IP", "Destination_IP", "Timestamp"]:
    print(f"   {'✔' if c not in df_full.columns else '⚠️'} {c}: "
          f"{'ausente (bueno)' if c not in df_full.columns else 'PRESENTE — revisar'}")

# Cachear
df_full.cache()

## 2. Análisis de calidad global

Barrido sobre las columnas numéricas buscando:
1. **Nulos** (`NULL`).
2. **NaN** (distinto de nulo, requiere `isnan()`).
3. **Infinitos** (`+inf`, `-inf`) — división por duración cero.
4. **Negativos imposibles** en magnitudes (duraciones, bytes, paquetes).

In [ ]:
# ============================================
# Barrido de calidad (una sola pasada con agg)
# ============================================
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, FloatType, IntegerType, LongType

# Definir num_cols (queda disponible para celdas siguientes)
num_cols = [f.name for f in df_full.schema.fields
            if isinstance(f.dataType, (DoubleType, FloatType, IntegerType, LongType))]
print(f"Columnas numéricas: {len(num_cols)}\n")

# Construir una lista de agregaciones (todas se ejecutan en un solo job)
exprs = []
for c in num_cols:
    safe = c.replace("/", "_").replace(".", "_").replace(" ", "_")
    exprs.append(F.sum(F.when(F.isnull(F.col(c)), 1).otherwise(0)).alias(f"{safe}__null"))
    exprs.append(F.sum(F.when(F.isnan(F.col(c)),  1).otherwise(0)).alias(f"{safe}__nan"))
    exprs.append(F.sum(F.when(F.col(c) == float("inf"),  1).otherwise(0)).alias(f"{safe}__pinf"))
    exprs.append(F.sum(F.when(F.col(c) == float("-inf"), 1).otherwise(0)).alias(f"{safe}__ninf"))
    exprs.append(F.sum(F.when(F.col(c) < 0, 1).otherwise(0)).alias(f"{safe}__neg"))

# Procesar en bloques de 15 columnas para no sobrecargar el plan
problemas = []
print("=== Barrido de calidad ===\n")
print(f"{'Columna':<40} {'Nulos':>8} {'NaN':>8} {'+inf':>8} {'-inf':>8} {'Neg':>8}")
print("-" * 80)

bloque = 15
for i in range(0, len(num_cols), bloque):
    cols_bloque = num_cols[i:i+bloque]
    exprs_bloque = []
    for c in cols_bloque:
        safe = c.replace("/", "_").replace(".", "_").replace(" ", "_")
        exprs_bloque.append(F.sum(F.when(F.isnull(F.col(c)), 1).otherwise(0)).alias(f"{safe}__null"))
        exprs_bloque.append(F.sum(F.when(F.isnan(F.col(c)),  1).otherwise(0)).alias(f"{safe}__nan"))
        exprs_bloque.append(F.sum(F.when(F.col(c) == float("inf"),  1).otherwise(0)).alias(f"{safe}__pinf"))
        exprs_bloque.append(F.sum(F.when(F.col(c) == float("-inf"), 1).otherwise(0)).alias(f"{safe}__ninf"))
        exprs_bloque.append(F.sum(F.when(F.col(c) < 0, 1).otherwise(0)).alias(f"{safe}__neg"))

    r = df_full.agg(*exprs_bloque).collect()[0]

    for c in cols_bloque:
        safe = c.replace("/", "_").replace(".", "_").replace(" ", "_")
        n_null = r[f"{safe}__null"] or 0
        n_nan  = r[f"{safe}__nan"]  or 0
        n_pinf = r[f"{safe}__pinf"] or 0
        n_ninf = r[f"{safe}__ninf"] or 0
        n_neg  = r[f"{safe}__neg"]  or 0

        if any([n_null, n_nan, n_pinf, n_ninf, n_neg]):
            print(f"{c:<40} {n_null:>8} {n_nan:>8} {n_pinf:>8} {n_ninf:>8} {n_neg:>8}")
            problemas.append({"columna": c, "nulos": n_null, "nan": n_nan,
                              "pinf": n_pinf, "ninf": n_ninf, "negativos": n_neg})

print(f"\n✅ Columnas con problemas: {len(problemas)}")

## 3. Limpieza global

**Estrategia:**

| Tipo | Tratamiento |
|---|---|
| Duraciones negativas | → 0 |
| Ratios con NaN/inf/negativo | → mediana de la columna (ignorando los problemáticos) |
| Otras numéricas con problemas | → mediana |

**Por qué mediana:** distribuciones con colas de hasta 2×10⁹; la media se contamina con outliers.

In [ ]:
# ============================================
# Limpieza global (autocontenida: no depende de celdas anteriores)
# ============================================
from pyspark.sql.functions import col, when, isnan, isnull
from pyspark.sql.types import DoubleType, FloatType, IntegerType, LongType

# Reconstruir num_cols por si acaso
num_cols = [f.name for f in df_full.schema.fields
            if isinstance(f.dataType, (DoubleType, FloatType, IntegerType, LongType))]

dur_cols   = [c for c in df_full.columns if "Duration" in c]
ratio_cols = [c for c in num_cols if c not in dur_cols]

def mediana_limpia(df, columna):
    """Mediana ignorando NaN, null, ±inf y negativos."""
    return df.filter(
        (~isnan(col(columna))) &
        (~isnull(col(columna))) &
        (col(columna) != float("inf")) &
        (col(columna) != float("-inf")) &
        (col(columna) >= 0)
    ).approxQuantile(columna, [0.5], 0.01)[0]

# Calcular medianas solo para las columnas que sabemos que tienen problemas
# (el resto no las necesita; así ahorramos tiempo)
columnas_con_problemas = set()
try:
    columnas_con_problemas = {p["columna"] for p in problemas}
except NameError:
    # Si la celda 6 no se ejecutó, hacemos un set razonable de candidatas
    columnas_con_problemas = {
        "Flow_Bytes/s", "Flow_Packets/s",
        "Fwd_Packets/s", "Bwd_Packets/s",
        "Flow_IAT_Mean", "Flow_IAT_Std", "Flow_IAT_Max", "Flow_IAT_Min",
        "Fwd_IAT_Total", "Fwd_IAT_Mean", "Fwd_IAT_Std", "Fwd_IAT_Max", "Fwd_IAT_Min",
        "Bwd_IAT_Total", "Bwd_IAT_Mean", "Bwd_IAT_Std", "Bwd_IAT_Max", "Bwd_IAT_Min"
    }

print("=== Calculando medianas limpias ===\n")
medianas = {}
for c in ratio_cols:
    if c in columnas_con_problemas:
        m = mediana_limpia(df_full, c)
        medianas[c] = m
        print(f"  {c:<40} → mediana: {m:,.4f}")

# Aplicar limpieza
print("\n=== Aplicando limpieza ===\n")
df_clean = df_full

# 1. Duraciones negativas → 0
for c in dur_cols:
    df_clean = df_clean.withColumn(c, when(col(c) < 0, 0).otherwise(col(c)))

# 2. Ratios con problemas → mediana
for c in ratio_cols:
    if c in medianas:
        df_clean = df_clean.withColumn(
            c,
            when(
                isnull(col(c)) |
                isnan(col(c)) |
                (col(c) == float("inf")) |
                (col(c) == float("-inf")) |
                (col(c) < 0),
                medianas[c]
            ).otherwise(col(c))
        )

print("✅ Limpieza aplicada")

In [ ]:
# ============================================
# Verificación post-limpieza
# ============================================
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, FloatType, IntegerType, LongType

# Reconstruir num_cols
num_cols = [f.name for f in df_clean.schema.fields
            if isinstance(f.dataType, (DoubleType, FloatType, IntegerType, LongType))]
dur_cols = [c for c in df_clean.columns if "Duration" in c]

print("=== Verificación post-limpieza ===\n")

restantes = 0
bloque = 15
for i in range(0, len(num_cols), bloque):
    cols_bloque = num_cols[i:i+bloque]
    exprs = []
    for c in cols_bloque:
        safe = c.replace("/", "_").replace(".", "_").replace(" ", "_")
        exprs.append(F.sum(F.when(F.isnull(F.col(c)), 1).otherwise(0)).alias(f"{safe}__null"))
        exprs.append(F.sum(F.when(F.isnan(F.col(c)),  1).otherwise(0)).alias(f"{safe}__nan"))
        exprs.append(F.sum(F.when(F.col(c) == float("inf"),  1).otherwise(0)).alias(f"{safe}__pinf"))
        exprs.append(F.sum(F.when(F.col(c) == float("-inf"), 1).otherwise(0)).alias(f"{safe}__ninf"))
        if c not in dur_cols:
            exprs.append(F.sum(F.when(F.col(c) < 0, 1).otherwise(0)).alias(f"{safe}__neg"))

    r = df_clean.agg(*exprs).collect()[0]
    columnas_row = r.__fields__  # lista de nombres de campos

    for c in cols_bloque:
        safe = c.replace("/", "_").replace(".", "_").replace(" ", "_")
        n_null = r[f"{safe}__null"] or 0
        n_nan  = r[f"{safe}__nan"]  or 0
        n_pinf = r[f"{safe}__pinf"] or 0
        n_ninf = r[f"{safe}__ninf"] or 0

        # Solo existe la clave __neg si la columna no es de duración
        neg_key = f"{safe}__neg"
        n_neg = (r[neg_key] or 0) if neg_key in columnas_row else 0

        tot = n_null + n_nan + n_pinf + n_ninf + n_neg
        if tot > 0:
            print(f"  ⚠️ {c}: null={n_null} nan={n_nan} +inf={n_pinf} -inf={n_ninf} neg={n_neg}")
            restantes += tot

if restantes == 0:
    print("✅ Dataset limpio: 0 valores problemáticos")
else:
    print(f"\n⚠️ Quedan {restantes} valores problemáticos")

df_clean.cache()
print(f"\nFilas tras limpieza: {df_clean.count():,}")

## 4. Distribución de clases

Dato crítico: condiciona la estrategia de balanceo y validación.

In [ ]:
# ============================================
# Distribución de clases
# ============================================
from pyspark.sql.functions import col, desc, round as spark_round

print("=== Distribución de clases ===\n")
total = df_clean.count()

dist = df_clean.groupBy("Label").count() \
    .orderBy(desc("count")) \
    .withColumn("porcentaje", spark_round(col("count") / total * 100, 4))

dist.show(truncate=False, n=30)

conteos = {r["Label"]: r["count"] for r in dist.collect()}
max_clase = max(conteos, key=conteos.get)
min_clase = min(conteos, key=conteos.get)
print(f"Total clases:       {len(conteos)}")
print(f"Clase mayoritaria:  {max_clase} ({conteos[max_clase]:,})")
print(f"Clase minoritaria:  {min_clase} ({conteos[min_clase]:,})")
print(f"Ratio desbalance:   {conteos[max_clase]/conteos[min_clase]:.1f}×")

## 5. Análisis de fuga de información

El dataset ya viene sin `Flow_ID`, `Source_IP`, `Destination_IP` ni `Timestamp`. El único sospechoso es `Destination_Port`: algunos ataques usan puertos exclusivos (21, 22, 444).

In [ ]:
# ============================================
# Confirmación: identificadores ausentes
# ============================================
print("=== Identificadores ===\n")
for c in ["Flow_ID", "Source_IP", "Destination_IP"]:
    if c in df_clean.columns:
        n = df_clean.select(c).distinct().count()
        print(f"⚠️ {c} → {n:,} valores únicos (¡revisar!)")
    else:
        print(f"✔ {c}: ausente")

ts_cols = [c for c in df_clean.columns
           if "timestamp" in c.lower() or "time" in c.lower()]
if ts_cols:
    print(f"⚠️ Columnas tipo timestamp: {ts_cols}")
else:
    print("✔ Timestamp: ausente")

print("\n💡 El análisis de fuga se centra en Destination_Port.")

In [ ]:
# ============================================
# Análisis de fuga por Destination_Port
# ============================================
from pyspark.sql.functions import col, desc

print("=== Puerto más frecuente por clase ===\n")

conteos_clase = {r["Label"]: r["count"]
                 for r in df_clean.groupBy("Label").count().collect()}
total_benign = conteos_clase.get("BENIGN", 0)

for clase in sorted(conteos_clase.keys()):
    top = df_clean.filter(col("Label") == clase) \
        .groupBy("Destination_Port").count() \
        .orderBy(desc("count")).limit(1).collect()

    if not top:
        continue

    puerto = top[0]["Destination_Port"]
    count = top[0]["count"]
    pct_clase = count / conteos_clase[clase] * 100

    benign_con_puerto = df_clean.filter(
        (col("Label") == "BENIGN") & (col("Destination_Port") == puerto)
    ).count()
    pct_benign = (benign_con_puerto / total_benign * 100) if total_benign else 0

    if pct_clase > 30 and pct_benign < 1:
        flag = "⚠️ FUGA ARTIFICIAL"
    elif pct_clase > 30:
        flag = "⚠️ dominante"
    elif pct_clase > 10:
        flag = "⚠️"
    else:
        flag = "  "

    print(f"{flag:22s} {clase:30s} → puerto {puerto:>6}: "
          f"{pct_clase:5.1f}% clase | BENIGN: {pct_benign:5.2f}%")

## 6. Visualización

In [ ]:
# ============================================
# Distribución de clases (muestra para pandas)
# ============================================
import matplotlib.pyplot as plt
import seaborn as sns

pdf = df_clean.select("Label").sample(fraction=0.01, seed=42).toPandas()

plt.figure(figsize=(10, 7))
sns.countplot(y="Label", data=pdf,
              order=pdf["Label"].value_counts().index,
              palette="viridis")
plt.title("Distribución de clases (muestra 1%)")
plt.xlabel("Nº de flujos")
plt.tight_layout()
plt.show()

## 7. Guardar Parquet limpio

- `cic-ids2017-parquet` → trazabilidad.
- `cic-ids2017-parquet-clean` → para modelado.

In [ ]:
# ============================================
# Guardar Parquet limpio
# ============================================
print(f"Guardando en: {PARQUET_CLEAN}")
print("Puede tardar 2-4 minutos...")

df_clean.write.mode("overwrite").parquet(PARQUET_CLEAN)

print("\n✅ Parquet limpio guardado")

df_check = spark.read.parquet(PARQUET_CLEAN)
print(f"Filas:    {df_check.count():,}")
print(f"Columnas: {len(df_check.columns)}")
df_check.select("Label", "Flow_Duration", "Flow_Bytes/s").show(5)

## 8. Decisiones para el modelado

| Columna | Decisión |
|---|---|
| `Flow_ID`, IPs, `Timestamp` | Ya ausentes |
| `Destination_Port` | Mantener + experimento con/sin |
| `Flow_Bytes/s`, `Flow_Packets/s` | Mantener, considerar log1p |

In [ ]:
# ============================================
# Diario y commit
# ============================================
%cd /content/sentry

from datetime import datetime
fecha = datetime.now().strftime("%Y-%m-%d")

entrada = f"""

---

## {fecha} — EDA final y Parquet limpio

**Qué probé:**
- Barrido de calidad sobre columnas numéricas.
- Limpieza global (mediana para ratios, 0 para duraciones negativas).
- Análisis de fuga: identificadores ausentes + análisis de Destination_Port.
- Parquet limpio en `cic-ids2017-parquet-clean`.

**Qué encontré:**
- Columnas con problemas: (rellenar).
- Identificadores: ausentes.
- Fugas artificiales confirmadas: (rellenar).

**Decisiones:**
- Mantener Destination_Port + experimento comparativo en Semana 2.
- Aplicar log1p a Flow_Bytes/s y Flow_Packets/s.
- Guardar Parquet limpio separado del raw.

**Próximos pasos:**
- Semana 2: pipeline de preprocesado.
"""

with open("docs/diario.md", "a") as f:
    f.write(entrada)

print("✅ Diario actualizado")
!tail -25 docs/diario.md

!git add .
!git commit -m "EDA final: limpieza global, análisis de fuga y Parquet limpio"
!git push

In [ ]:
%cd /content/sentry

entrada = """

---

## 2026-09-27 — Cierre del EDA final (datos reales)

**Dataset limpio:**
- 2.830.743 filas × 79 columnas tras limpieza (0 valores problemáticos).
- Parquet limpio guardado en `cic-ids2017-parquet-clean`.
- Identificadores (`Flow_ID`, `Source_IP`, `Destination_IP`, `Timestamp`) ausentes desde el raw.

**Columnas con problemas de calidad tratadas (11):**
`Flow_Bytes/s`, `Flow_Packets/s`, `Flow_IAT_Mean/Max/Min`, `Fwd_IAT_Min`,
`Fwd_Header_Length34`, `Bwd_Header_Length`, `Fwd_Header_Length55`,
`Init_Win_bytes_forward/backward`, `min_seg_size_forward`.
Tratamiento: mediana calculada ignorando NaN/inf/negativos.

**Distribución de clases (15 clases):**
- BENIGN: 2.273.097 (80.30%)
- DoS Hulk: 231.073 (8.16%)
- PortScan: 158.930 (5.61%)
- DDoS: 128.027 (4.52%)
- DoS GoldenEye: 10.293 (0.36%)
- FTP-Patator: 7.938 (0.28%)
- SSH-Patator: 5.897 (0.21%)
- DoS slowloris: 5.796 (0.20%)
- DoS Slowhttptest: 5.499 (0.19%)
- Bot: 1.966 (0.07%)
- Web Attack - Brute Force: 1.507 (0.05%)
- Web Attack - XSS: 652 (0.02%)
- Infiltration: 36 (0.001%)
- Web Attack - Sql Injection: 21 (0.0007%)
- Heartbleed: 11 (0.0004%)

Ratio de desbalance máximo: 206.645×.

**Fugas artificiales confirmadas por Destination_Port:**
- Bot → 8080 (BENIGN: 0.06%)
- FTP-Patator → 21 (BENIGN: 0.23%)
- SSH-Patator → 22 (BENIGN: 0.48%)
- Heartbleed → 444 (BENIGN: 0.00%)
- Infiltration → 444 (BENIGN: 0.00%)

**Puertos dominantes legítimos (no son fuga):**
- DoS Hulk, DoS GoldenEye, DoS slowloris, DoS Slowhttptest, DDoS,
  Web Attack (Brute Force, XSS, Sql Injection) → 80, con BENIGN también en 80 (10.37%).

**Decisiones tomadas para el modelado:**
- `Destination_Port`: mantener + experimento comparativo con/sin para cuantificar el impacto de la fuga artificial.
- Clases con <50 muestras (Heartbleed, Infiltration, Sql Injection): no viables para entrenamiento supervisado directo. Decisión pendiente en Semana 2 (agrupar o excluir).
- Aplicar `log1p` a features de cola larga (`Flow_Bytes/s`, `Flow_Packets/s`, IATs) antes del escalado.

**Próximos pasos:**
- Semana 2: diseño del target (binario + familias), pipeline de preprocesado, split y balanceo.
"""

with open("docs/diario.md", "a") as f:
    f.write(entrada)

!git add .
!git commit -m "Cerrar Semana 1: EDA final con datos reales y decisiones"
!git push